#### 1. 라이브러리 임포트

In [1]:
import random
import numpy as np
from pathlib import Path
import pandas as pd
import tensorflow as tf
from transformers import BertTokenizerFast, TFBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

I0000 00:00:1780516079.225658   55715 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780516079.254555   55715 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780516080.062713   55715 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


#### 3. 데이터 로드

In [2]:
# 데이터 경로
dataset_name = "final_data"
parquet_path = Path(f"{dataset_name}.parquet")

# Parquet 파일 읽기
df = pd.read_parquet(parquet_path)

# 데이터 크기 확인
print(f"데이터 크기: {df.shape}")
df.sample(5)

데이터 크기: (70000, 7)


,review_text,fake,basic_linguistic_list,readability_list,sentiment_list,behavioral_list,clean_text
42345,Guilt free pizza. Â So thin and so tasty!,1,"[11.0, 9.0, 2.0, 41.0, 31.0, 0.0, 7.0, 3.0, 3....","[0.0, 98.8675, 0.5872222222222234, 1.8, 10.877...","[-0.04999999999999999, 0.8250000000000001, 0.0...","[18.0, 84.0, 4.9411764705882355, 15.6183809509...",guilt free pizza. so thin and so tasty
68645,"There are so many great, detailed and informat...",1,"[123.0, 100.0, 11.0, 562.0, 431.0, 2.0, 80.0, ...","[6.627428350798422, 87.1588392857143, 4.363035...","[0.46875, 0.5375, 3.0, 0.0, 4.0, 0.0, 2.0]","[2.0, 8.0, 8.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0]",there are so many great detailed and informati...
16305,So after seeing this place on triple D and rea...,0,"[256.0, 196.0, 14.0, 1006.0, 770.0, 5.0, 142.0...","[6.67199483983844, 80.86691159586682, 5.821094...","[0.2841477272727272, 0.5060416666666667, 3.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0]",so after seeing this place on triple d and rea...
51067,When I ordered Grilled Salmon from the menu. T...,1,"[184.0, 134.0, 14.0, 763.0, 596.0, 11.0, 96.0,...","[8.384062270229773, 79.58212374581943, 4.79637...","[0.3284965034965035, 0.631118881118881, 0.0, 0...","[2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0]",when i ordered grilled salmon from the menu. t...
20188,I was not let down by Toto Ramen. Â We tried t...,1,"[90.0, 65.0, 7.0, 326.0, 251.0, 5.0, 46.0, 9.0...","[7.957251820313856, 80.27153846153847, 4.36989...","[0.3841269841269842, 0.553968253968254, 0.0, 0...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0]",i was not let down by toto ramen. we tried the...


#### 4. Train / Validation / Test 분할

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["fake"]), df["fake"].astype("float32"), test_size=0.2, stratify=df["fake"], random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.125, stratify=y_train, random_state=42
)

#### 6. Tokenize

In [5]:
MAX_LEN = 256

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

train_enc = tokenizer(list(X_train["clean_text"]), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="np")
val_enc = tokenizer(list(X_val["clean_text"]), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="np")
test_enc = tokenizer(list(X_test["clean_text"]), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="np")

train_inputs = {"input_ids": train_enc["input_ids"], "attention_mask": train_enc["attention_mask"]}
val_inputs = {"input_ids": val_enc["input_ids"], "attention_mask": val_enc["attention_mask"]}
test_inputs = {"input_ids": test_enc["input_ids"], "attention_mask": test_enc["attention_mask"]}

#### 7. Bert 함수

In [6]:
bert = TFBertModel.from_pretrained('bert-base-uncased')
bert.trainable = False

ids = tf.keras.Input((MAX_LEN,), dtype=tf.int32, name='input_ids')
mask = tf.keras.Input((MAX_LEN,), dtype=tf.int32, name='attention_mask')

x = bert(ids, attention_mask=mask).last_hidden_state
x = tf.keras.layers.Flatten()(x)

x = tf.keras.layers.Dense(2048, activation='gelu')(x)
x = tf.keras.layers.Dense(1024, activation='gelu')(x)
x = tf.keras.layers.Dense(512, activation='gelu')(x)
x = tf.keras.layers.Dense(256, activation='gelu')(x)
x = tf.keras.layers.Dense(128, activation='gelu')(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)

I0000 00:00:1780516100.811941   55715 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22149 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a Ber

#### 8. 모델 생성

In [7]:
model = tf.keras.Model(inputs={'input_ids': ids, 'attention_mask': mask}, outputs=out)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 256)]                0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 256)]                0         []                            
 )                                                                                                
                                                                                                  
 tf_bert_model (TFBertModel  TFBaseModelOutputWithPooli   1094822   ['input_ids[0][0]',           
 )                           ngAndCrossAttentions(last_   40         'attention_mask[0][0]']      
                             hidden_state=(None, 256, 7                                       

#### 9. 모델 컴파일

In [8]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])

#### 10. 콜백 설정

In [9]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

#### 11. 모델 학습

In [10]:
history = model.fit(x={"input_ids": train_enc["input_ids"], "attention_mask": train_enc["attention_mask"]}, y=y_train, 
                    validation_data=({"input_ids": val_enc["input_ids"], "attention_mask": val_enc["attention_mask"]}, y_val), epochs=20, batch_size=32, callbacks=callbacks, verbose=1)

Epoch 1/20


I0000 00:00:1780516116.292293   55904 service.cc:153] XLA service 0x72ecbae6c880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780516116.292310   55904 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4090, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1780516116.295545   55904 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780516116.308179   55904 cuda_dnn.cc:461] Loaded cuDNN version 92300
I0000 00:00:1780516116.346827   55904 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1532/1532 [==============================] - 168s 106ms/step - loss: 0.6461 - accuracy: 0.6299 - val_loss: 0.6263 - val_accuracy: 0.6480
Epoch 2/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.5978 - accuracy: 0.6753 - val_loss: 0.6326 - val_accuracy: 0.6567
Epoch 3/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.5382 - accuracy: 0.7201 - val_loss: 0.6674 - val_accuracy: 0.6569
Epoch 4/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.4601 - accuracy: 0.7703 - val_loss: 0.7625 - val_accuracy: 0.6354
Epoch 5/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.3823 - accuracy: 0.8168 - val_loss: 0.9215 - val_accuracy: 0.6341
Epoch 6/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.3096 - accuracy: 0.8570 - val_loss: 1.0228 - val_accuracy: 0.6291


#### 12. 예측 및 성능 계산

In [11]:
y_pred_prob = model.predict({"input_ids": test_enc["input_ids"], "attention_mask": test_enc["attention_mask"]}, batch_size=32)
y_pred = (y_pred_prob > 0.5).astype(int)

print(f"acc  : {accuracy_score(y_test, y_pred):.4f}")
print(f"prec : {precision_score(y_test, y_pred):.4f}")
print(f"rec  : {recall_score(y_test, y_pred):.4f}")
print(f"f1   : {f1_score(y_test, y_pred):.4f}")

438/438 [==============================] - 32s 70ms/step
acc  : 0.6415
prec : 0.6833
rec  : 0.5274
f1   : 0.5953
